In [ ]:
import os
print(os.listdir('/content'))

['.config', 'Cleaned_Dataset.csv', 'sample_data']


In [ ]:
import pandas as pd
df = pd.read_csv('Cleaned_Dataset.csv')
df = df.dropna(subset=['profit'])
print(df.shape)

(51257, 21)


In [ ]:
print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())

order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_name     0
segment           0
state             0
country           0
market            0
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
quantity          0
discount          0
profit            0
shipping_cost     0
order_priority    0
year              0
dtype: int64
Duplicates: 0


In [ ]:
df = df.drop_duplicates()
print(df.shape)

(51271, 21)


In [ ]:
print(df['sales'].dtype)
print(df['sales'][df['sales'].str.contains(',')].head())

object
27    1,648
43    3,029
58    2,574
83    2,479
84    2,174
Name: sales, dtype: object


In [ ]:
df['sales'] = df['sales'].str.replace(',', '').astype(float)
print(df['sales'].dtype)

float64


In [ ]:
print(df['order_date'].sample(8, random_state=1))

27367    22-08-2013
46248    17-10-2014
33737    31-12-2013
1929       2/5/2011
24785    17-06-2013
10109      7/3/2012
38968      3/6/2014
35484      2/3/2014
Name: order_date, dtype: object


In [ ]:
for col in ['order_date', 'ship_date']:
    df[col] = pd.to_datetime(df[col].str.replace('-', '/'), format='%d/%m/%Y')

print(df['order_date'].dtype)
print(df[['order_date', 'ship_date']].head())

datetime64[ns]
  order_date  ship_date
0 2011-01-01 2011-01-06
1 2011-01-01 2011-01-08
2 2011-01-01 2011-01-05
3 2011-01-01 2011-01-05
4 2011-01-01 2011-01-08


In [ ]:
df['customer_name'] = df['customer_name'].fillna('Unknown')
print(df['customer_name'].isna().sum())

0


In [ ]:
df['customer_name'] = df['customer_name'].fillna('Unknown')
print(df['customer_name'].isna().sum())

0


In [ ]:
df['product_name'] = df['product_name'].str.strip().str.replace(r'\s+', ' ', regex=True)

In [ ]:
print(df.shape)
print(df.isna().sum().sum())
print(df.dtypes)

(51271, 21)
14
order_id                  object
order_date        datetime64[ns]
ship_date         datetime64[ns]
ship_mode                 object
customer_name             object
segment                   object
state                     object
country                   object
market                    object
region                    object
product_id                object
category                  object
sub_category              object
product_name              object
sales                    float64
quantity                   int64
discount                 float64
profit                   float64
shipping_cost            float64
order_priority            object
year                       int64
dtype: object


In [ ]:
print((df['quantity'] <= 0).sum())
print(((df['discount'] < 0) | (df['discount'] > 1)).sum())
print((df['shipping_cost'] < 0).sum())
print((df['sales'] <= 0).sum())

0
0
0
1


In [ ]:
print(df.shape)
print(df.isna().sum().sum())
print(df.duplicated().sum())

(51271, 21)
14
0


In [ ]:
df.to_csv('Cleaned_Dataset.csv', index=False)
from google.colab import files
files.download('Cleaned_Dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(df.shape)
print(df['profit'].isna().sum())

(51257, 21)
0


In [ ]:
# Task 1 Level 2
import sqlite3

conn = sqlite3.connect('primeor.db')
df.to_sql('orders', conn, if_exists='replace', index=False)

print(pd.read_sql('SELECT COUNT(*) AS total_rows FROM orders', conn))

   total_rows
0       51257


In [ ]:
print(pd.read_sql('SELECT * FROM orders LIMIT 5', conn))

          order_id  order_date   ship_date       ship_mode    customer_name  \
0     AG-2011-2040  2011-01-01  2011-01-06  Standard Class  Toby Braunhardt   
1    IN-2011-47883  2011-01-01  2011-01-08  Standard Class      Joseph Holt   
2     HU-2011-1220  2011-01-01  2011-01-05    Second Class    Annie Thurman   
3  IT-2011-3647632  2011-01-01  2011-01-05    Second Class     Eugene Moren   
4    IN-2011-47883  2011-01-01  2011-01-08  Standard Class      Joseph Holt   

       segment            state    country  market   region  ...  \
0     Consumer      Constantine    Algeria  Africa   Africa  ...   
1     Consumer  New South Wales  Australia    APAC  Oceania  ...   
2     Consumer         Budapest    Hungary    EMEA     EMEA  ...   
3  Home Office        Stockholm     Sweden      EU    North  ...   
4     Consumer  New South Wales  Australia    APAC  Oceania  ...   

          category sub_category                 product_name  sales  quantity  \
0  Office Supplies      Storage    

In [ ]:
query = """
SELECT product_name, SUM(profit) AS total_profit
FROM orders
GROUP BY product_name
ORDER BY total_profit DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,product_name,total_profit
0,Canon imageCLASS 2200 Advanced Copier,25199.9280
1,"Cisco Smart Phone, Full Size",17238.5206
2,"Motorola Smart Phone, Full Size",17027.1130
3,"Hoover Stove, Red",11807.9690
4,"Sauder Classic Bookcase, Traditional",10672.0730
5,"Harbour Creations Executive Leather Armchair, ...",10427.3260
6,"Nokia Smart Phone, Full Size",9938.1955
7,"Cisco Smart Phone, with Caller ID",9786.6408
8,"Nokia Smart Phone, with Caller ID",9465.3257
9,"Belkin Router, USB",8955.0180


In [ ]:
query = """
SELECT customer_name, SUM(sales) AS total_sales
FROM orders
GROUP BY customer_name
ORDER BY total_sales DESC
LIMIT 10
"""
pd.read_sql(query, conn)


,customer_name,total_sales
0,Tom Ashbrook,40489.0
1,Tamara Chand,37453.0
2,Greg Tran,35552.0
3,Christopher Conant,35187.0
4,Sean Miller,35131.0
5,Bart Watters,32315.0
6,Natalie Fritzler,31778.0
7,Fred Hopkins,30404.0
8,Jane Waco,30288.0
9,Hunter Lopez,30246.0


In [ ]:
query = """
SELECT region, SUM(sales) AS total_sales
FROM orders
GROUP BY region
ORDER BY total_sales DESC
"""
pd.read_sql(query, conn)

,region,total_sales
0,Central,2821394.0
1,South,1600584.0
2,North,1248121.0
3,Oceania,1098217.0
4,Southeast Asia,883314.0
5,North Asia,846991.0
6,EMEA,804920.0
7,Africa,783261.0
8,Central Asia,752839.0
9,West,724311.0


In [ ]:
query = """
SELECT category, AVG(profit) AS avg_profit
FROM orders
GROUP BY category
ORDER BY avg_profit DESC
"""
pd.read_sql(query, conn)

,category,avg_profit
0,Technology,65.455996
1,Furniture,29.004043
2,Office Supplies,16.576218


In [ ]:
query = """
SELECT category, AVG(discount) AS avg_discount
FROM orders
GROUP BY category
ORDER BY avg_discount DESC
"""
pd.read_sql(query, conn)

,category,avg_discount
0,Furniture,0.168034
1,Office Supplies,0.137412
2,Technology,0.135369


In [ ]:
query = """
SELECT order_id, product_name, sales, discount, profit
FROM orders
WHERE profit < 0
ORDER BY profit ASC
LIMIT 20
"""
pd.read_sql(query, conn)

,order_id,product_name,sales,discount,profit
0,CA-2013-108196,Cubify CubeX 3D Printer Double Head Print,4500.0,0.70,-6599.9780
1,TU-2013-9400,"Motorola Smart Phone, Cordless",3085.0,0.60,-4088.3760
2,US-2014-168116,Cubify CubeX 3D Printer Triple Head Print,8000.0,0.50,-3839.9904
3,CA-2011-169019,GBC DocuBind P400 Electric Binding System,2178.0,0.80,-3701.8928
4,CA-2014-134845,Lexmark MX611dhe Monochrome Laser Printer,2550.0,0.70,-3399.9800
5,IT-2013-3695467,"Hoover Stove, White",3400.0,0.50,-3059.8200
6,ID-2013-12295,"Apple Smart Phone, Full Size",3499.0,0.50,-3009.4350
7,US-2014-122714,Ibico EPK-21 Electric Binding System,1890.0,0.80,-2929.4845
8,LH-2014-5390,"Barricks Conference Table, Rectangular",2171.0,0.70,-2750.2800
9,CA-2012-147830,Cubify CubeX 3D Printer Double Head Print,1800.0,0.70,-2639.9912


In [ ]:
query = """
SELECT COUNT(*) AS loss_rows
FROM orders
WHERE profit < 0
"""
pd.read_sql(query, conn)

,loss_rows
0,12538


In [ ]:
query = """
SELECT strftime('%Y-%m', order_date) AS month, SUM(sales) AS total_sales
FROM orders
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

,month,total_sales
0,2011-01,98376.0
1,2011-02,90901.0
2,2011-03,144451.0
3,2011-04,116835.0
4,2011-05,146115.0
5,2011-06,215214.0
6,2011-07,115518.0
7,2011-08,207570.0
8,2011-09,290230.0
9,2011-10,199070.0


In [ ]:
query = """
SELECT market, SUM(sales) AS total_revenue
FROM orders
GROUP BY market
ORDER BY total_revenue DESC
"""
pd.read_sql(query, conn)

,market,total_revenue
0,APAC,3581361.0
1,EU,2936867.0
2,US,2295756.0
3,LATAM,2163876.0
4,EMEA,804920.0
5,Africa,783261.0
6,Canada,66332.0


In [ ]:
query = """
SELECT sub_category, SUM(sales) AS total_sales, SUM(profit) AS total_profit
FROM orders
GROUP BY sub_category
ORDER BY total_sales DESC
LIMIT 5
"""
pd.read_sql(query, conn)

,sub_category,total_sales,total_profit
0,Phones,1706515.0,216689.12660
1,Copiers,1508370.0,258174.34514
2,Chairs,1500339.0,141676.50410
3,Bookcases,1463615.0,161703.18750
4,Storage,1126285.0,108508.95680


In [ ]:
query = """
SELECT ship_mode, COUNT(*) AS total_orders
FROM orders
GROUP BY ship_mode
ORDER BY total_orders DESC
"""
pd.read_sql(query, conn)

,ship_mode,total_orders
0,Standard Class,30756
1,Second Class,10301
2,First Class,7503
3,Same Day,2697


## SQL Insights

- Highest revenue market: ...
- Least profitable category: ...
- Most used shipping mode: ...



**Query 1 – Top 10 profitable products**
The Canon imageCLASS 2200 Advanced Copier is the most profitable product (about 25,200 profit). Five of the top 10 are smartphones from Cisco, Motorola and Nokia, so phones are a major profit driver.

**Query 2 – Top 10 customers by sales**
Tom Ashbrook is the top customer (about 40,489 in sales), followed by Tamara Chand (37,453). The top 10 customers together contribute only about 2.7% of total sales, so revenue is spread across many customers rather than a few.

**Query 3 – Region-wise total sales**
Central is the strongest region with about 2.82 million in sales (22.3% of the total), followed by South (12.7%) and North (9.9%). Canada is the weakest region with about 66 thousand.

**Query 4 – Category-wise average profit**
Technology has the highest average profit per order line (about 65.5), then Furniture (29.0). Office Supplies is the least profitable category with about 16.6 per order line.

**Query 5 – Highest discount category**
Furniture receives the highest average discount (about 16.8%), compared with Office Supplies (13.7%) and Technology (13.5%). This may be one reason why Furniture earns less profit than Technology.

**Query 6 – Orders with negative profit**
About 12,538 rows (24.5% of all rows) are loss-making, with a total loss of roughly 920 thousand. Loss-making orders have an average discount of about 45%, compared with about 4% for profitable orders, so heavy discounting is strongly linked to losses.

**Query 7 – Monthly sales trend**
Sales grow steadily from 2011 to 2014 (yearly sales rise from about 2.26 million to 4.29 million). The highest month is November 2014 (about 555 thousand), and sales are generally stronger in November and December.

**Query 8 – Market-wise revenue**
APAC generates the highest revenue (about 3.58 million, 28.4%), followed by EU (23.2%) and US (18.2%). Canada is the smallest market with only about 0.5%.

**Query 9 – Top-performing sub-categories**
Phones lead in sales (about 1.71 million), followed by Copiers and Chairs. However, Copiers earn the highest profit (about 258 thousand), higher than Phones (about 217 thousand), so top sales do not always mean top profit.

**Query 10 – Ship mode usage**
Standard Class is the most commonly used shipping mode (30,756 orders, about 60% of all orders). Same Day is the least used (about 5%).

In [ ]:
df.to_csv('Final_Cleaned_Dataset.csv', index=False)
from google.colab import files
files.download('Final_Cleaned_Dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>